# Synthetic Q&A Generation for Document Fine-Tuning

Generate a fine-tuning dataset from a source document (PDF or markdown), filter on quality, and fine-tune a small model.

End-to-end:
1. Extract text from PDF or load markdown
2. Chunk a large source + run parallel Foundry datagen jobs (works around per-source saturation at ~100-150 pairs per single job)
3. LLM-judge quality filter (drops fragmented / off-topic / empty pairs)
4. Split into train / val / test
5. Score the base model on the test set (correctness via LLM judge)
6. Submit one fine-tuning job (winning hyperparameters: 1 epoch, lr=0.5 — low and slow to prevent overfitting)
7. Deploy the fine-tuned model
8. Score it on the same test set and report the lift

**Realistic expectations**: Q&A from a reference document is **hard** for small SFT — the questions usually require *factual recall* the small model lacks. Format-style tasks (enforce a tone, output structure, persona) ship reliably; pure-knowledge tasks often need RAG instead. The notebook will report the honest lift and suggest alternatives if SFT doesn't help.

**Cost**: ~$5–15 per run. **Time**: ~30–60 minutes.

This notebook is **fully self-contained** — no external skill scripts required. All helpers (chunking, parallel datagen, quality filter, evaluator) are defined inline.

## 1. Setup & Configuration

In [ ]:
import os, json, time, random, re, subprocess, sys, io
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
BASE_URL         = os.environ["OPENAI_BASE_URL"]
API_KEY          = os.environ["AZURE_OPENAI_API_KEY"]

SOURCE_DOC       = Path("fixtures/sample_source.md")  # REPLACE with your own PDF or .md
TEACHER_MODEL    = "gpt-4.1"        # used for Q&A generation and quality-filter judge
STUDENT_MODEL    = "gpt-4.1-mini"   # the model we are fine-tuning

WORK             = Path("./run").resolve()
WORK.mkdir(exist_ok=True)

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"Source:    {SOURCE_DOC} ({SOURCE_DOC.stat().st_size:,} bytes)")
print(f"Teacher:   {TEACHER_MODEL}")
print(f"Student:   {STUDENT_MODEL}")


## 2. Inline helpers: chunked Foundry datagen

Why chunk? Foundry's `SimpleQnA` recipe saturates at ~100-150 unique Q&A pairs per source file regardless of `max_samples` — the teacher self-deduplicates aggressively. To get more data from a large reference, chunk the source into N pieces and run N parallel jobs.

**Note on TPM headroom**: each parallel job reads the chunk into the teacher's context many times per generated pair. Concurrency × chunk-size × passes must fit under the teacher's TPM quota. For a 500K-TPM `gpt-4.1`, concurrency=2 works with chunks ≤150KB. Lower concurrency if you hit rate limits.

In [ ]:
def chunk_text(text, n_chunks, overlap_chars=1000):
    """Split text into n_chunks roughly-equal pieces with paragraph-boundary
    snapping. Small overlap between adjacent chunks helps the teacher see
    context boundaries.
    """
    if n_chunks <= 1: return [text]
    target = len(text) // n_chunks
    chunks, pos = [], 0
    for i in range(n_chunks):
        start = max(0, pos - (overlap_chars if i > 0 else 0))
        end = min(len(text), pos + target + overlap_chars)
        if i == n_chunks - 1: end = len(text)
        if end < len(text):
            nl = text.rfind("\n\n", pos, end)
            if nl > pos: end = nl
        chunks.append(text[start:end])
        pos = end
    return chunks


def submit_qna_datagen_for_chunk(*, file_id, teacher, max_samples,
                                 output_name, project_endpoint, timeout_s=1800):
    """Submit one SimpleQnA datagen job and return the downloaded JSONL path."""
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import (
        DataGenerationJob, DataGenerationJobInputs,
        DataGenerationJobScenario, DataGenerationJobOutputOptions,
        SimpleQnADataGenerationJobOptions, FileDataGenerationJobSource,
        DataGenerationModelOptions,
    )
    from azure.identity import DefaultAzureCredential

    project = AIProjectClient(endpoint=project_endpoint, credential=DefaultAzureCredential())
    options = SimpleQnADataGenerationJobOptions(
        max_samples=max_samples,
        model_options=DataGenerationModelOptions(model=teacher),
    )
    job = DataGenerationJob(inputs=DataGenerationJobInputs(
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        sources=[FileDataGenerationJobSource(file_id=file_id)],
        options=options,
        output_options=DataGenerationJobOutputOptions(name=output_name),
    ))
    created = project.datasets.create_generation_job(body=job)
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        j = project.datasets.get_generation_job(name=created.id)
        if str(j.status).endswith("SUCCEEDED"): break
        if str(j.status).endswith(("FAILED", "CANCELLED")): raise RuntimeError(f"{output_name}: {j.status}")
        time.sleep(15)
    else:
        raise TimeoutError(f"{output_name} did not complete")
    out = WORK / f"{output_name}_dg.jsonl"
    blob = b""
    for fid in j.outputs[0].file_ids:
        blob += client.files.content(file_id=fid).read()
    out.write_bytes(blob)
    return out


def chunk_and_generate_qna(source_text, n_chunks, teacher, max_samples_per_chunk,
                            project_endpoint, concurrency=2, out_path=None):
    """Chunk + parallel datagen + concatenate. Works around Foundry's per-source
    saturation (~100-150 unique pairs per single job) by running N jobs.

    NOTE: each parallel job reads the chunk into the teacher's context many
    times per generated pair. Concurrency × chunk-size × passes must fit
    under the teacher's TPM quota. For a 500K-TPM gpt-4.1, concurrency=2
    works with chunks ≤150KB. Lower concurrency if you hit rate limits.
    """
    chunks = chunk_text(source_text, n_chunks)
    print(f"Source split into {n_chunks} chunks (min={min(len(c) for c in chunks):,}, max={max(len(c) for c in chunks):,})")

    # Upload chunks
    file_ids = []
    for i, ch in enumerate(chunks):
        buf = io.BytesIO(ch.encode("utf-8"))
        f = client.files.create(file=(f"chunk-{i:02d}.txt", buf), purpose="user_data")
        for _ in range(30):
            f = client.files.retrieve(file_id=f.id)
            if f.status == "processed": break
            time.sleep(2)
        file_ids.append(f.id)
        print(f"  chunk {i}: uploaded {f.id}")

    # Run datagen in parallel
    produced = []
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futures = {
            ex.submit(submit_qna_datagen_for_chunk,
                      file_id=fid, teacher=teacher,
                      max_samples=max_samples_per_chunk,
                      output_name=f"chunk-{i:02d}",
                      project_endpoint=project_endpoint): i
            for i, fid in enumerate(file_ids)
        }
        for fut in as_completed(futures):
            i = futures[fut]
            try:
                p = fut.result()
                n = sum(1 for line in p.open(encoding="utf-8") if line.strip())
                print(f"  chunk {i}: ✅ {n} rows")
                produced.append(p)
            except Exception as e:
                print(f"  chunk {i}: ❌ {str(e)[:100]}")

    # Concatenate
    out = out_path or (WORK / "merged.jsonl")
    total = 0
    with out.open("w", encoding="utf-8") as fout:
        for p in sorted(produced):
            with p.open(encoding="utf-8") as fin:
                for line in fin:
                    if line.strip():
                        fout.write(line); total += 1

    # Cleanup uploaded chunks
    for fid in file_ids:
        try: client.files.delete(fid)
        except Exception: pass

    print(f"\nMerged {len(produced)}/{len(chunks)} chunks -> {out.name} ({total} rows)")
    return out


## 3. Inline helper: LLM-judge quality filter

Even with a good teacher, synthetic Q&A often includes a few rows that are fragmented, refusals, or off-topic. The judge scores each pair on three axes (`non_fragmented`, `non_empty`, `on_topic`) on a 1-5 scale; rows below threshold get dropped.

In [ ]:
JUDGE_PROMPT = """You are a quality reviewer for AI fine-tuning data. Score this prompt/response pair on three axes (1-5 each). Return ONLY a JSON object.

Prompt:
{prompt}

Response:
{response}

Score each axis 1 (worst) to 5 (best):
- non_fragmented: 5 = complete coherent answer; 1 = cut off mid-sentence or truncated
- non_empty: 5 = substantive, actually answers; 1 = blank, refuses to engage
- on_topic: 5 = directly addresses the prompt; 1 = unrelated or hallucinated

Return JSON: {{"non_fragmented": <int>, "non_empty": <int>, "on_topic": <int>, "reason": "1-line note"}}
"""

def quality_filter_jsonl(input_path, judge_model, threshold=4, concurrency=4, out_path=None):
    """LLM-judge each prompt/response pair on 3 axes; drop rows below threshold.
    Returns path to filtered JSONL."""
    with open(input_path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    def _extract(row):
        msgs = row.get("messages") or []
        user = next((m.get("content") or "" for m in msgs if m.get("role") == "user"), "")
        asst = next((m.get("content") or "" for m in msgs if m.get("role") == "assistant"), "")
        return user, asst

    def _score(i_row):
        i, row = i_row
        user, asst = _extract(row)
        if not user or not asst: return i, {"non_fragmented": 1, "non_empty": 1, "on_topic": 1}
        try:
            resp = client.chat.completions.create(
                model=judge_model,
                messages=[{"role": "user", "content": JUDGE_PROMPT.format(prompt=user[:3000], response=asst[:3000])}],
                temperature=0.0, max_completion_tokens=200,
            )
            text = (resp.choices[0].message.content or "").strip()
            m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
            if not m: return i, {"non_fragmented": 0, "non_empty": 0, "on_topic": 0}
            s = json.loads(m.group())
            return i, {"non_fragmented": int(s.get("non_fragmented", 0)),
                       "non_empty": int(s.get("non_empty", 0)),
                       "on_topic": int(s.get("on_topic", 0))}
        except Exception:
            return i, {"non_fragmented": 0, "non_empty": 0, "on_topic": 0}

    keep = [True] * len(rows); scored = 0
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futures = [ex.submit(_score, (i, r)) for i, r in enumerate(rows)]
        for fut in as_completed(futures):
            i, sc = fut.result(); scored += 1
            if min(sc["non_fragmented"], sc["non_empty"], sc["on_topic"]) < threshold:
                keep[i] = False
            if scored % 25 == 0:
                print(f"  scored {scored}/{len(rows)}")

    out = out_path or (WORK / "filtered.jsonl")
    kept = 0
    with out.open("w", encoding="utf-8") as f:
        for i, row in enumerate(rows):
            if keep[i]:
                f.write(json.dumps(row) + "\n"); kept += 1
    print(f"  Kept {kept}/{len(rows)} (dropped {len(rows) - kept})")
    return out


## 4. Generate training data from your source document

In [ ]:
# Extract text if PDF, otherwise use as-is
if SOURCE_DOC.suffix.lower() == ".pdf":
    from pypdf import PdfReader  # pip install pypdf
    text_path = WORK / (SOURCE_DOC.stem + ".txt")
    reader = PdfReader(SOURCE_DOC)
    text_path.write_text("\n\n".join(p.extract_text() or "" for p in reader.pages), encoding="utf-8")
    print(f"Extracted {len(reader.pages)} pages -> {text_path.name} ({text_path.stat().st_size:,} bytes)")
else:
    text_path = SOURCE_DOC

source_text = text_path.read_text(encoding="utf-8")
print(f"Source: {len(source_text):,} chars, ~{len(source_text.split()):,} words")

# Pick number of chunks: aim for ~150KB chunks for safe TPM, 1 chunk for short docs
n_chunks = max(1, min(15, len(source_text) // 150_000))
print(f"Generating with {n_chunks} chunks @ 100 samples/chunk = ~{n_chunks * 100} unique Q&A pairs target")

MERGED = chunk_and_generate_qna(
    source_text=source_text,
    n_chunks=n_chunks,
    teacher=TEACHER_MODEL,
    max_samples_per_chunk=100,
    project_endpoint=PROJECT_ENDPOINT,
    concurrency=2,   # safe for 500K TPM teacher; lower if you hit rate limits
    out_path=WORK / "merged.jsonl",
)

with open(MERGED, encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]
print(f"\nGenerated: {len(data)} rows")
print("First row preview:")
print(json.dumps(data[0]["messages"], indent=2)[:600])


## 5. Quality-filter the generated data

In [ ]:
# Filter on quality. Drops rows where the judge scores <4 on any axis.
FILTERED = quality_filter_jsonl(
    input_path=MERGED,
    judge_model=TEACHER_MODEL,
    threshold=4,
    concurrency=4,
    out_path=WORK / "filtered.jsonl",
)
with open(FILTERED, encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]
print(f"\nQuality-filtered: {len(data)} rows")


## 6. Split into train / val / test

In [ ]:
rng = random.Random(42)
indices = list(range(len(data))); rng.shuffle(indices)
n_train = int(0.8 * len(indices)); n_val = int(0.1 * len(indices))
train_idx = indices[:n_train]; val_idx = indices[n_train:n_train+n_val]; test_idx = indices[n_train+n_val:]

TRAIN_PATH = WORK / "train.jsonl"; VAL_PATH = WORK / "val.jsonl"; TEST_PATH = WORK / "test.jsonl"
for path, idxs in [(TRAIN_PATH, train_idx), (VAL_PATH, val_idx), (TEST_PATH, test_idx)]:
    with path.open("w", encoding="utf-8") as f:
        for i in idxs: f.write(json.dumps(data[i]) + "\n")

print(f"  train: {len(train_idx)} rows")
print(f"  val:   {len(val_idx)} rows")
print(f"  test:  {len(test_idx)} rows")


## 7. Baseline evaluation via the Foundry evals SDK

We use `azure-ai-evaluation.evaluate()` as the driver, with a custom correctness evaluator that asks the teacher model to score each answer against the gold reference on a 1-10 scale.

In [ ]:
# Build the data file the SDK expects: each row has columns for target + evaluator
EVAL_DATA_PATH = WORK / "eval_data.jsonl"
with open(TEST_PATH, encoding="utf-8") as fin, open(EVAL_DATA_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip(): continue
        row = json.loads(line)
        msgs = row["messages"]
        user = next((m.get("content") or "" for m in msgs if m.get("role") == "user"), "")
        asst = next((m.get("content") or "" for m in msgs if m.get("role") == "assistant"), "")
        fout.write(json.dumps({"query": user, "ground_truth": asst}) + "\n")
print(f"Eval data: {EVAL_DATA_PATH.name}")


def make_target(model_name):
    """Returns a callable the evaluate() SDK uses to query a model row-by-row."""
    def _target(*, query, **kwargs):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": query}],
                temperature=0.0, max_completion_tokens=1024,
            )
            return {"response": resp.choices[0].message.content or ""}
        except Exception as e:
            return {"response": "", "error": str(e)[:200]}
    return _target


# Custom LLM-judge evaluator scoring on correctness (vs ground truth)
JUDGE_EVAL_PROMPT = """Score this answer for correctness on a 1-10 scale. Return ONLY JSON.

Question: {query}

Reference (gold) answer:
{ground_truth}

Model answer:
{response}

Score: 10 = matches reference essentially; 7-9 = correct but different wording / partial; 4-6 = some right elements but significant errors; 1-3 = mostly wrong.

Return: {{"correctness": <int 1-10>}}
"""

def correctness_evaluator(*, query, ground_truth, response, **kwargs):
    try:
        r = client.chat.completions.create(
            model=TEACHER_MODEL,
            messages=[{"role": "user", "content": JUDGE_EVAL_PROMPT.format(
                query=query, ground_truth=ground_truth[:1500], response=response[:1500])}],
            temperature=0.0, max_completion_tokens=80,
        )
        text = (r.choices[0].message.content or "").strip()
        m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
        score = int(json.loads(m.group()).get("correctness", 0)) if m else 0
    except Exception:
        score = 0
    return {"correctness_score": score, "correctness_pass": 1.0 if score >= 8 else 0.0}


from azure.ai.evaluation import evaluate
print(f"\nBaseline ({STUDENT_MODEL}) evaluation...")
baseline_results = evaluate(
    data=str(EVAL_DATA_PATH),
    target=make_target(STUDENT_MODEL),
    evaluators={"correctness": correctness_evaluator},
    output_path=str(WORK / "baseline_eval_results.json"),
)
baseline_combined = baseline_results["metrics"].get("correctness.correctness_score", 0)
baseline_pass = baseline_results["metrics"].get("correctness.correctness_pass", 0) * 100
print(f"  Baseline: correctness={baseline_combined:.2f}/10  pass_rate={baseline_pass:.1f}%")


## 8. Submit the fine-tuning job

Winning hyperparameters from prior experiments: **1 epoch, LR multiplier 0.5** for Q&A datasets in the ~400 train row range. Low and slow because the val/best ratio shows easy overfitting at default LR=1.0 + 2-3 epochs.

In [ ]:
print("Uploading train + val files...")
with open(TRAIN_PATH, "rb") as fh:
    train_file = client.files.create(file=(TRAIN_PATH.name, fh), purpose="fine-tune")
with open(VAL_PATH, "rb") as fh:
    val_file = client.files.create(file=(VAL_PATH.name, fh), purpose="fine-tune")

for f in (train_file, val_file):
    for _ in range(30):
        f = client.files.retrieve(file_id=f.id)
        if f.status == "processed": break
        time.sleep(2)
    print(f"  {f.id} status={f.status}")

# Winning HPs: 1 epoch, LR multiplier 0.5 — low and slow to prevent overfitting
# on the ~400-row Q&A dataset. For other Q&A datasets, sweep a few combinations.
print("\nSubmitting fine-tuning job...")
ft_job = client.fine_tuning.jobs.create(
    model=STUDENT_MODEL,
    training_file=train_file.id,
    validation_file=val_file.id,
    method={"type": "supervised"},
    hyperparameters={"n_epochs": 1, "learning_rate_multiplier": 0.5},
    suffix="docqna-demo",
    extra_body={"trainingType": "globalStandard"},
)
print(f"  Job: {ft_job.id}  status={ft_job.status}")


## 9. Monitor training

In [ ]:
print(f"Monitoring job {ft_job.id}...\n")
last_seen_step = -1
while True:
    job = client.fine_tuning.jobs.retrieve(ft_job.id)
    events = list(client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_job.id, limit=10))
    for e in reversed(events):
        msg = e.message or ""
        if "Step " in msg and ":" in msg:
            try:
                step = int(msg.split("Step ")[1].split(":")[0])
                if step > last_seen_step:
                    print(f"  {time.strftime('%H:%M:%S')}  {msg[:90]}")
                    last_seen_step = step
            except Exception: pass
    if job.status in ("succeeded", "failed", "cancelled"):
        print(f"\nFinal status: {job.status}")
        if job.status != "succeeded":
            print(f"Error: {job.error.message if job.error else '(none)'}")
            raise RuntimeError(f"Fine-tuning {job.status}")
        FT_MODEL_ID = job.fine_tuned_model
        print(f"Fine-tuned model: {FT_MODEL_ID}")
        break
    time.sleep(30)


## 10. Deploy the fine-tuned model

In [ ]:
m = re.match(r"https://([^.]+)\.openai\.azure\.com", BASE_URL)
ACCOUNT_NAME    = m.group(1)
SUBSCRIPTION_ID = os.environ.get("AZURE_SUBSCRIPTION_ID") or input("Azure subscription id: ")
RESOURCE_GROUP  = os.environ.get("AZURE_RESOURCE_GROUP") or input("Azure resource group: ")

DEPLOY_NAME = "docqna-ft-demo"
deploy_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}"
    f"/deployments/{DEPLOY_NAME}?api-version=2024-10-01"
)
body = {"sku": {"name": "GlobalStandard", "capacity": 100},
        "properties": {"model": {"format": "OpenAI", "name": FT_MODEL_ID, "version": "1"}}}

import urllib.request, urllib.error
token = subprocess.check_output(["az", "account", "get-access-token", "--query", "accessToken", "-o", "tsv"]).decode().strip()
req = urllib.request.Request(deploy_url, method="PUT",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    data=json.dumps(body).encode())
try: urllib.request.urlopen(req); print(f"Deployment submitted: {DEPLOY_NAME}")
except urllib.error.HTTPError as e: print(f"Deploy returned: {e.code} {e.reason}")

print("Waiting for deployment to become inferenceable (~3-5 min)...")
for i in range(20):
    try:
        client.chat.completions.create(model=DEPLOY_NAME, messages=[{"role":"user","content":"hi"}], max_completion_tokens=5)
        print(f"\n  Ready after {i*30}s"); break
    except Exception:
        time.sleep(30); print(".", end="", flush=True)


## 11. Evaluate the fine-tuned model and compare

In [ ]:
print(f"Fine-tuned ({STUDENT_MODEL}) evaluation...")
ft_results = evaluate(
    data=str(EVAL_DATA_PATH),
    target=make_target(DEPLOY_NAME),
    evaluators={"correctness": correctness_evaluator},
    output_path=str(WORK / "ft_eval_results.json"),
)
ft_combined = ft_results["metrics"].get("correctness.correctness_score", 0)
ft_pass = ft_results["metrics"].get("correctness.correctness_pass", 0) * 100

print()
print("-" * 60)
print(f"  {'Model':<35}  {'Correctness':>12}  {'Pass Rate':>10}")
print(f"  {'-'*35}  {'-'*12}  {'-'*10}")
print(f"  Baseline ({STUDENT_MODEL}){' '*(35-19-len(STUDENT_MODEL))}  {baseline_combined:>12.2f}  {baseline_pass:>9.1f}%")
print(f"  Fine-tuned ({STUDENT_MODEL}){' '*(35-21-len(STUDENT_MODEL))}  {ft_combined:>12.2f}  {ft_pass:>9.1f}%")

lift = (ft_combined - baseline_combined) / baseline_combined * 100 if baseline_combined > 0 else 0
print(f"\n  Lift:  {lift:+.1f}%")

if lift >= 5:
    print(f"  Fine-tuning improved correctness by {lift:+.1f}% -- ship the FT model.")
else:
    print(f"  Lift below 5% threshold.")
    print(f"  Q&A from a reference document is genuinely hard for small SFT — pure factual recall")
    print(f"  is something small models lack and SFT alone can't bridge. Consider:")
    print(f"  - RAG (let the small model look up the doc at inference) instead of/alongside SFT")
    print(f"  - Larger student (e.g. gpt-4.1)")
    print(f"  - Format-style tasks where SFT shines: enforcing structured output, terminology, persona")


## Cleanup

```python
req = urllib.request.Request(deploy_url, method="DELETE", headers={"Authorization": f"Bearer {token}"})
urllib.request.urlopen(req)
client.files.delete(train_file.id); client.files.delete(val_file.id)
```

## Bring your own document

Replace `SOURCE_DOC` at the top with the path to your PDF or markdown. PDFs are extracted with `pypdf`. For very large documents the chunking auto-scales (1 chunk per ~150KB).

## Dependencies

```
pip install openai>=2.0 azure-ai-projects>=2.2.0 azure-identity>=1.21 azure-ai-evaluation>=1.0 pypdf>=4.0
```